# Reproducción **leaky de Cheuque** sobre **KOZYRIEV** — robustness check del protocolo

**Versión 1 de las dos versiones deep para Kozyriev** (la otra es `deep_kozyriev_h3.ipynb` = 5-core + protocolo justo).

Es el **mismo cuadernillo que `cheuque_ucsd_inicial.ipynb` pero sobre Kozyriev**: filtro **denso** (≥ ítems/usuario y ≥ usuarios/ítem), label `playtime≥5h`, **playtime como feature** (la fuga), eval **re-ranking por-usuario** de los ítems de test → reproduce el régimen ~0,94–0,99 + la columna **sin fuga**. ALS hace full-ranking como contraste.

**Para qué:** mostrar que el NDCG≈0,99 de los deep es **artefacto de protocolo** (label binario + re-ranking por-usuario + fuga de playtime), **no** algo específico de UCSD. Si reaparece en Kozyriev, el argumento de la *reconciliación con Cheuque* es robusto.

> ⚠️ **Caveat clave de Kozyriev:** su señal es **reviews** (`recommendations.csv`), no *ownership* como UCSD → los usuarios tienen **pocas** interacciones, así que el **≥100 ítems/usuario de Cheuque puede COLAPSAR** el dataset. La celda del filtro imprime conteos a varios umbrales y, si el de Cheuque deja <300 usuarios, hace **fallback al subset más denso viable** (declarándolo). El filtro de Cheuque es sobre compras; acá es sobre reviews → análogo, no idéntico.

> El `hours` de Kozyriev ya está **en horas** (UCSD venía en minutos) → umbral = 5 directo.


## 0. Setup (instalación + seed + config)

In [1]:
# En Colab. Instala deps (mismas que cheuque_ucsd_inicial + kagglehub para descargar Kozyriev).
!pip install -q implicit deepctr-torch kagglehub torch pandas numpy scikit-learn pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.0 MB/s eta 0:00:00


In [2]:
import os, ast, gzip, urllib.request, math, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

SEED = 42
import random
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")  # evita contención de BLAS en ALS

# Config (defaults = paper; sobreescribible por env para correr rápido en CPU)
def _cfg(name, d): return int(os.environ.get(name, d))
ALS_FACTORS = _cfg("CHEUQUE_ALS_FACTORS", 500); ALS_ITERS = _cfg("CHEUQUE_ALS_ITERS", 300)
ALS_FOLDS = _cfg("CHEUQUE_ALS_FOLDS", 5)
DEEP_EPOCHS = _cfg("CHEUQUE_DEEP_EPOCHS", 15); DEEP_FOLDS = _cfg("CHEUQUE_DEEP_FOLDS", 3)
DEEP_BATCH = _cfg("CHEUQUE_DEEP_BATCH", 512)
print("device:", DEVICE, "| pandas", pd.__version__, "| numpy", np.__version__)
print(f"config ALS(f={ALS_FACTORS},it={ALS_ITERS},folds={ALS_FOLDS}) "
      f"DEEP(ep={DEEP_EPOCHS},folds={DEEP_FOLDS},batch={DEEP_BATCH})")

device: cuda | pandas 2.2.2 | numpy 2.0.2
config ALS(f=500,it=300,folds=5) DEEP(ep=15,folds=3,batch=512)


In [3]:
# Config Kozyriev (dataset + imports extra para la descarga kagglehub)
from pathlib import Path
import zipfile, shutil, csv, glob, json, subprocess
import kagglehub
SLUG         = 'antonkozyriev/game-recommendations-on-steam'
FRONKON_SLUG = 'fronkongames/steam-games-dataset'
print('config Kozyriev OK | kagglehub', getattr(kagglehub, '__version__', '?'))

config Kozyriev OK | kagglehub 1.0.2


## 1. Descargar Kozyriev + FronkonGames

In [4]:
def _find_file(root, *names):
    cand = {n.lower() for n in names}
    for p in Path(root).rglob('*'):
        if p.is_file() and p.name.lower() in cand:
            return p
    raise FileNotFoundError(f'No encontré {names} bajo {root}')

def _maybe_unzip(p, *wanted):
    with open(p, 'rb') as f:
        if f.read(4) != b'PK\x03\x04':
            return p
    ext = p.parent / (p.stem + '_extracted'); ext.mkdir(exist_ok=True)
    with zipfile.ZipFile(p) as z:
        names = z.namelist(); target = None
        for w in wanted:
            target = next((n for n in names if Path(n).name.lower() == w.lower()), None)
            if target: break
        if target is None:
            target = next((n for n in names if n.lower().endswith(('.csv', '.json'))), names[0])
        out = ext / Path(target).name
        if not out.exists() or out.stat().st_size == 0:
            with z.open(target) as s, open(out, 'wb') as d: shutil.copyfileobj(s, d)
        return out

def fetch_file(slug, *filenames):
    for fn in filenames:
        try:
            p = Path(kagglehub.dataset_download(slug, path=fn))
            p = p if p.is_file() else _find_file(p, fn)
            return _maybe_unzip(p, *filenames)
        except Exception:
            continue
    return _find_file(Path(kagglehub.dataset_download(slug)), *filenames)

def read_csv_robust(path, **kw):
    for enc in ('utf-8', 'utf-8-sig', 'cp1252', 'latin-1'):
        try:
            return pd.read_csv(path, encoding=enc, **kw)
        except (UnicodeDecodeError, UnicodeError):
            continue
    return pd.read_csv(path, encoding='latin-1', encoding_errors='replace', **kw)

t0 = time.time()
games_csv = fetch_file(SLUG, 'games.csv')
meta_json = fetch_file(SLUG, 'games_metadata.json')
recs_csv  = fetch_file(SLUG, 'recommendations.csv')
fg_csv    = fetch_file(FRONKON_SLUG, 'games.csv')
print(f'Descargado en {time.time()-t0:.0f}s')

100%|██████████| 1.41M/1.41M [00:01<00:00, 1.37MB/s]

Extracting zip of games.csv...


100%|██████████| 5.47M/5.47M [00:01<00:00, 3.96MB/s]

Extracting zip of games_metadata.json...


100%|██████████| 580M/580M [00:31<00:00, 19.5MB/s]

Extracting zip of recommendations.csv...


100%|██████████| 383M/383M [00:28<00:00, 14.0MB/s]

Descargado en 98s


In [5]:
# ====== Carga Kozyriev: recommendations (dedup) + genres (FronkonGames) + positive_ratio + tags ======
def _pick_col(df, *al):
    low = {c.lower(): c for c in df.columns}
    for a in al:
        if a.lower() in low: return low[a.lower()]
    return None

# recommendations.csv -> pares UNICOS (user, app, hours, is_recommended)
parts = []
for chunk in pd.read_csv(recs_csv, usecols=['app_id', 'user_id', 'hours', 'is_recommended'],
                         chunksize=2_000_000,
                         dtype={'app_id':'int32','user_id':'int32','hours':'float32','is_recommended':'bool'}):
    parts.append(chunk)
recs_full = pd.concat(parts, ignore_index=True); del parts
recs_full = recs_full.drop_duplicates(['user_id', 'app_id'], keep='last', ignore_index=True)
print(f'pares unicos (user,app): {len(recs_full):,} | usuarios {recs_full.user_id.nunique():,} | juegos {recs_full.app_id.nunique():,}')

# FronkonGames -> genres por app (con reparacion del header 'DiscountDLC count')
with open(fg_csv, encoding='utf-8', errors='replace', newline='') as f:
    header = next(csv.reader(f))
fixed, rep = [], False
for h in header:
    if h.strip().lower() in ('discountdlc count', 'discount dlc count'):
        fixed += ['Discount', 'DLC count']; rep = True
    else: fixed.append(h)
if rep:
    fgdf = read_csv_robust(fg_csv, engine='python', on_bad_lines='skip', dtype=str, header=0, names=fixed)
else:
    fgdf = read_csv_robust(fg_csv, engine='python', on_bad_lines='skip', dtype=str, index_col=False)
_cg = _pick_col(fgdf, 'Genres', 'genres'); _ca = _pick_col(fgdf, 'AppID', 'app_id', 'appid', 'steam_appid')
def _splitg(x):
    if x is None or (isinstance(x, float) and pd.isna(x)): return []
    return [t.strip() for t in str(x).split(',') if t and t.strip().lower() != 'nan']
genre_map = {}
for a, g in zip(pd.to_numeric(fgdf[_ca], errors='coerce'), fgdf[_cg]):
    if pd.notna(a): genre_map[int(a)] = _splitg(g)

# games.csv -> positive_ratio (= metascore del PER en H3)
gfull = read_csv_robust(games_csv); gfull['app_id'] = pd.to_numeric(gfull['app_id'], errors='coerce')
gfull = gfull.dropna(subset=['app_id'])
_cr = _pick_col(gfull, 'positive_ratio', 'rating')
ratio_map = {int(a): (float(r) if pd.notna(r) else np.nan)
             for a, r in zip(gfull['app_id'], pd.to_numeric(gfull[_cr], errors='coerce'))}

# metadata json -> tags (fallback de genero)
tags_map = {}
with open(meta_json, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try: rec = json.loads(line)
        except json.JSONDecodeError: continue
        aid = rec.get('app_id')
        if aid is not None: tags_map[int(aid)] = [str(t).strip() for t in (rec.get('tags') or [])]
print(f'genres FronkonGames: {len(genre_map):,} apps | positive_ratio: '
      f'{int(np.sum([pd.notna(v) for v in ratio_map.values()])):,} apps')

def iterative_filter(df, min_user, min_item, ucol='user_id', icol='app_id'):
    cur = df
    while True:
        n0 = len(cur)
        uc = cur[ucol].value_counts(); ic = cur[icol].value_counts()
        cur = cur[cur[ucol].isin(uc[uc >= min_user].index) & cur[icol].isin(ic[ic >= min_item].index)]
        if len(cur) == n0 or len(cur) == 0:
            return cur

pares unicos (user,app): 41,154,773 | usuarios 13,781,059 | juegos 37,610
genres FronkonGames: 125,855 apps | positive_ratio: 50,872 apps


## 2. Filtro denso (Cheuque ≥200/ítem, ≥100/usuario) — diagnóstico + fallback

In [6]:
# Filtro denso de Cheuque: >=MIN_ITEM reviews/juego, >=MIN_USER juegos/usuario (iterativo).
# OJO Kozyriev = REVIEWS (no ownership): usuarios con pocas reviews -> el >=100/usuario puede COLAPSAR.
# Diagnostico a varios umbrales + fallback al subset mas denso viable (>=MIN_VIABLE_USERS).
MIN_USER_DENSE, MIN_ITEM_DENSE = 100, 200          # Cheuque exacto
MIN_VIABLE_USERS = 300                              # piso para que ALS/deep tengan sentido
print('Conteos por umbral (user>=, item>=):')
for mu, mi in [(100, 200), (50, 100), (20, 50), (10, 30), (5, 20)]:
    d = iterative_filter(recs_full[['user_id', 'app_id']], mu, mi)
    print(f'  ({mu:>3},{mi:>3}) -> {d.user_id.nunique():>7,} usuarios | {d.app_id.nunique():>6,} juegos | {len(d):>10,} interac')

dense = iterative_filter(recs_full, MIN_USER_DENSE, MIN_ITEM_DENSE)
if dense['user_id'].nunique() < MIN_VIABLE_USERS:
    print(f'\n[!] Cheuque exacto ({MIN_USER_DENSE},{MIN_ITEM_DENSE}) deja '
          f'{dense.user_id.nunique():,} usuarios (<{MIN_VIABLE_USERS}); Kozyriev es review-based.')
    for mu, mi in [(50, 100), (20, 50), (10, 30), (5, 20)]:
        d = iterative_filter(recs_full, mu, mi)
        if d['user_id'].nunique() >= MIN_VIABLE_USERS:
            MIN_USER_DENSE, MIN_ITEM_DENSE, dense = mu, mi, d
            print(f'    -> fallback al mas denso viable: ({mu},{mi})')
            break
dense = dense.reset_index(drop=True)
nU, nI, nN = dense.user_id.nunique(), dense.app_id.nunique(), len(dense)
print(f'\nDENSO usado ({MIN_USER_DENSE},{MIN_ITEM_DENSE}) -> {nU:,} usuarios | {nI:,} juegos | '
      f'{nN:,} interac | densidad {nN/(nU*nI)*100:.2f}%')
print('Cheuque/UCSD ref (OTRO dataset, ownership): 8.183 usuarios / 2.872 juegos / 9.14% '
      '-> comparar el PATRON (deep~0.94 vs ALS~0.33), no el numero absoluto')

Conteos por umbral (user>=, item>=):
  (100,200) ->       0 usuarios |      0 juegos |          0 interac
  ( 50,100) ->  28,311 usuarios |  4,865 juegos |  2,399,940 interac
  ( 20, 50) -> 205,697 usuarios | 11,944 juegos |  7,960,019 interac
  ( 10, 30) -> 674,126 usuarios | 17,611 juegos | 14,315,887 interac
  (  5, 20) -> 1,905,447 usuarios | 22,676 juegos | 22,286,960 interac

[!] Cheuque exacto (100,200) deja 0 usuarios (<300); Kozyriev es review-based.
    -> fallback al mas denso viable: (50,100)

DENSO usado (50,100) -> 28,311 usuarios | 4,865 juegos | 2,399,940 interac | densidad 1.74%
Cheuque/UCSD ref (OTRO dataset, ownership): 8.183 usuarios / 2.872 juegos / 9.14% -> comparar el PATRON (deep~0.94 vs ALS~0.33), no el numero absoluto


## 3. Features + label (`playtime≥5h`, con fuga)

In [7]:
# Features + label (playtime>=5h). Kozyriev: 'hours' YA esta EN HORAS (UCSD era minutos).
PLAYTIME_THRESH_H = 5
df = dense.copy()
df['playtime']   = df['hours'].astype('float32')
df['label']      = (df['playtime'] >= PLAYTIME_THRESH_H).astype(int)
print(f"label positivo (>= {PLAYTIME_THRESH_H}h): {df['label'].mean()*100:.1f}%")
df['playtime_h'] = df['playtime']                                   # feature dense (la fuga)
df['count']      = df.groupby('user_id')['app_id'].transform('size').astype('float32')
# reccount = #is_recommended por juego (analogo al RecCount de Cheuque)
_rc = recs_full.groupby('app_id')['is_recommended'].sum()
df['reccount']   = df['app_id'].map(_rc).fillna(0.0).astype('float32')
df['recommend']  = df['is_recommended'].astype(int)
# metascore = positive_ratio (fillna mediana)
df['metascore']  = df['app_id'].map(ratio_map).astype('float32')
df['metascore']  = df['metascore'].fillna(df['metascore'].median() if df['metascore'].notna().any() else 0.0)
# encoders user/item
u_ids = sorted(df['user_id'].unique()); i_ids = sorted(df['app_id'].unique())
uid2idx = {u: k for k, u in enumerate(u_ids)}; iid2idx = {a: k for k, a in enumerate(i_ids)}
df['user_idx'] = df['user_id'].map(uid2idx).astype(int); df['item_idx'] = df['app_id'].map(iid2idx).astype(int)
N_USERS, N_ITEMS = len(u_ids), len(i_ids)
# generos -> secuencia padded (fallback a tags) ; mismo esquema que cheuque_ucsd_inicial
def _glist(a):
    g = genre_map.get(a) or tags_map.get(a) or []
    return [str(x) for x in g]
gmap = df['app_id'].map(_glist)
all_genres = sorted({g for lst in gmap for g in lst})
genre2id = {g: k + 1 for k, g in enumerate(all_genres)}            # 0 = padding
VOCAB_GENRE = len(all_genres) + 1; MAXLEN_G = 3
def pad_g(lst):
    ids = [genre2id[g] for g in lst][:MAXLEN_G]
    return ids + [0] * (MAXLEN_G - len(ids))
df['genres_seq'] = gmap.apply(pad_g)
df['genres_len'] = gmap.apply(lambda l: max(1, min(len(l), MAXLEN_G)))
print(f'usuarios={N_USERS} items={N_ITEMS} generos={len(all_genres)} | df={df.shape}')

label positivo (>= 5h): 68.5%
usuarios=28311 items=4865 generos=282 | df=(2399940, 15)


## 4. ALS baseline (`implicit`) — full-ranking, 5-fold CV

In [8]:
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

def split_per_user(frame, test_frac=0.2, seed=0):
    """Split aleatorio por-usuario: ~test_frac de los items de cada usuario van a test."""
    rng = np.random.RandomState(seed)
    test_mask = np.zeros(len(frame), dtype=bool)
    for _, idx in frame.groupby("user_idx").indices.items():
        idx = np.array(idx)
        if len(idx) < 2:
            continue
        n_test = max(1, int(round(len(idx) * test_frac)))
        chosen = rng.choice(idx, size=min(n_test, len(idx) - 1), replace=False)
        test_mask[chosen] = True
    return frame[~test_mask], frame[test_mask]

def ndcg_map_at_k(ranked_items, relevant_set, k=10):
    ranked = ranked_items[:k]
    # NDCG
    dcg = sum(1.0 / math.log2(i + 2) for i, it in enumerate(ranked) if it in relevant_set)
    ideal = sum(1.0 / math.log2(i + 2) for i in range(min(len(relevant_set), k)))
    ndcg = dcg / ideal if ideal > 0 else 0.0
    # MAP (AP@k)
    hits, ap = 0, 0.0
    for i, it in enumerate(ranked):
        if it in relevant_set:
            hits += 1; ap += hits / (i + 1)
    ap = ap / min(len(relevant_set), k) if relevant_set else 0.0
    return ndcg, ap

def run_als_fold(frame, seed):
    train, test = split_per_user(frame, test_frac=0.2, seed=seed)
    # matriz usuario x item con horas
    ui = csr_matrix((train["playtime_h"].values + 1e-6,
                     (train["user_idx"].values, train["item_idx"].values)),
                    shape=(N_USERS, N_ITEMS))
    try:
        als = AlternatingLeastSquares(factors=ALS_FACTORS, regularization=0.01, alpha=40.0,
                                      iterations=ALS_ITERS, random_state=seed)
        als.fit(ui, show_progress=False)
    except TypeError:
        als = AlternatingLeastSquares(factors=ALS_FACTORS, regularization=0.01,
                                      iterations=ALS_ITERS, random_state=seed)
        als.fit((ui * 40.0), show_progress=False)
    # eval full-ranking
    test_by_user = test.groupby("user_idx")["item_idx"].apply(set).to_dict()
    ndcgs, maps = [], []
    for u, rel in test_by_user.items():
        ids, _ = als.recommend(u, ui[u], N=10, filter_already_liked_items=True)
        n, a = ndcg_map_at_k(list(ids), rel, 10)
        ndcgs.append(n); maps.append(a)
    return float(np.mean(ndcgs)), float(np.mean(maps))

t0 = time.time()
als_res = [run_als_fold(df, s) for s in range(ALS_FOLDS)]  # 5 folds (paper)
als_ndcg = np.mean([r[0] for r in als_res]); als_map = np.mean([r[1] for r in als_res])
print(f"ALS (5-fold, full-ranking)  NDCG@10={als_ndcg:.3f}  MAP@10={als_map:.3f}  "
      f"[{time.time()-t0:.0f}s]   (paper 0,332 / 0,107)")

ALS (5-fold, full-ranking)  NDCG@10=0.140  MAP@10=0.062  [1835s]   (paper 0,332 / 0,107)


## 5. FM / DeepFM / DeepNN (`deepctr-torch`) — per-user, leaky

In [9]:
from deepctr_torch.inputs import SparseFeat, DenseFeat, VarLenSparseFeat, get_feature_names
from deepctr_torch.models import DeepFM, WDL
from sklearn.model_selection import train_test_split

EMB = 32
DENSE_COLS_FULL = ["playtime_h", "count", "reccount", "metascore"]

def make_feature_cols(dense_cols):
    sparse = [SparseFeat("user_idx", N_USERS, embedding_dim=EMB),
              SparseFeat("item_idx", N_ITEMS, embedding_dim=EMB),
              SparseFeat("recommend", 2, embedding_dim=EMB)]  # FM exige misma dim en todos los sparse
    varlen = [VarLenSparseFeat(SparseFeat("genres_seq", VOCAB_GENRE, embedding_dim=EMB),
                               maxlen=MAXLEN_G, combiner="mean", length_name="genres_len")]
    dense = [DenseFeat(c, 1) for c in dense_cols]
    cols = sparse + dense + varlen
    return cols, cols  # linear y dnn usan las mismas features

def build_X(frame, feat_names, dense_cols):
    X = {}
    for n in feat_names:
        if n in ("genres_seq", "genres_len"): continue
        if n in frame.columns: X[n] = frame[n].values
    X["genres_seq"] = np.stack(frame["genres_seq"].values)
    X["genres_len"] = frame["genres_len"].values
    return X

def build_model(kind, linear_cols, dnn_cols):
    if kind == "FM":      # solo lineal + FM (sin DNN)
        return DeepFM(linear_cols, dnn_cols, dnn_hidden_units=(), task="binary", device=DEVICE)
    if kind == "DeepFM":  # FM + DNN (mejor config Tabla 3: 8x8)
        return DeepFM(linear_cols, dnn_cols, dnn_hidden_units=(8, 8), dnn_dropout=0.2, task="binary", device=DEVICE)
    if kind == "DeepNN":  # solo parte profunda (WDL con wide vacío)
        return WDL([], dnn_cols, dnn_hidden_units=(32, 32), dnn_dropout=0.2, task="binary", device=DEVICE)
    raise ValueError(kind)

def eval_per_user(test_frame, scores, k=10):
    ev = test_frame[["user_idx", "label"]].copy(); ev["score"] = scores
    ndcgs, maps = [], []
    for _, g in ev.groupby("user_idx"):
        g = g.sort_values("score", ascending=False).head(k)
        rel = g["label"].values
        # NDCG con relevancia binaria
        dcg = sum(r / math.log2(i + 2) for i, r in enumerate(rel))
        n_rel = int(test_frame.loc[test_frame["user_idx"] == g["user_idx"].iloc[0], "label"].sum()) if len(g) else 0
        ideal = sum(1.0 / math.log2(i + 2) for i in range(min(n_rel, k)))
        ndcgs.append(dcg / ideal if ideal > 0 else 0.0)
        hits, ap = 0, 0.0
        for i, r in enumerate(rel):
            if r > 0:
                hits += 1; ap += hits / (i + 1)
        maps.append(ap / min(n_rel, k) if n_rel > 0 else 0.0)
    return float(np.mean(ndcgs)), float(np.mean(maps))

def run_deep_fold(frame, kind, dense_cols, seed, epochs=None, batch=None):
    epochs = DEEP_EPOCHS if epochs is None else epochs
    batch = DEEP_BATCH if batch is None else batch
    lin, dnn = make_feature_cols(dense_cols)
    feat_names = get_feature_names(lin + dnn)
    tr, te = train_test_split(frame, test_size=0.2, random_state=seed, stratify=frame["label"])
    Xtr, ytr = build_X(tr, feat_names, dense_cols), tr["label"].values
    Xte = build_X(te, feat_names, dense_cols)
    m = build_model(kind, lin, dnn)
    m.compile("adam", "binary_crossentropy", metrics=["auc"])
    m.fit(Xtr, ytr, batch_size=batch, epochs=epochs, verbose=0)
    sc = m.predict(Xte, batch_size=4096).reshape(-1)
    return eval_per_user(te, sc, 10)

def run_deep(kind, dense_cols, folds=None, **kw):
    folds = DEEP_FOLDS if folds is None else folds
    res = [run_deep_fold(df, kind, dense_cols, s, **kw) for s in range(folds)]
    return np.mean([r[0] for r in res]), np.mean([r[1] for r in res])

deep_res = {}
for kind in ["FM", "DeepFM", "DeepNN"]:
    t0 = time.time()
    n, mp = run_deep(kind, DENSE_COLS_FULL)
    deep_res[kind] = (n, mp)
    print(f"{kind:7s} (3-fold, per-user)  NDCG@10={n:.3f}  MAP@10={mp:.3f}  [{time.time()-t0:.0f}s]")

cuda
Train on 1919952 samples, validate on 0 samples, 3750 steps per epoch
cuda
Train on 1919952 samples, validate on 0 samples, 3750 steps per epoch
cuda
Train on 1919952 samples, validate on 0 samples, 3750 steps per epoch
FM      (3-fold, per-user)  NDCG@10=0.999  MAP@10=0.998  [1515s]
cuda
Train on 1919952 samples, validate on 0 samples, 3750 steps per epoch
cuda
Train on 1919952 samples, validate on 0 samples, 3750 steps per epoch
cuda
Train on 1919952 samples, validate on 0 samples, 3750 steps per epoch
DeepFM  (3-fold, per-user)  NDCG@10=0.998  MAP@10=0.997  [1656s]
cuda
Train on 1919952 samples, validate on 0 samples, 3750 steps per epoch
cuda
Train on 1919952 samples, validate on 0 samples, 3750 steps per epoch
cuda
Train on 1919952 samples, validate on 0 samples, 3750 steps per epoch
DeepNN  (3-fold, per-user)  NDCG@10=0.999  MAP@10=0.998  [1260s]


## 6. Variante honesta (sin fuga) — quita `playtime` de las features

In [10]:
DENSE_COLS_NOLEAK = ["count", "reccount", "metascore"]  # sin playtime_h
deep_res_noleak = {}
for kind in ["FM", "DeepFM", "DeepNN"]:
    t0 = time.time()
    n, mp = run_deep(kind, DENSE_COLS_NOLEAK)
    deep_res_noleak[kind] = (n, mp)
    print(f"{kind:7s} SIN FUGA  NDCG@10={n:.3f}  MAP@10={mp:.3f}  [{time.time()-t0:.0f}s]")

cuda
Train on 1919952 samples, validate on 0 samples, 3750 steps per epoch
cuda
Train on 1919952 samples, validate on 0 samples, 3750 steps per epoch
cuda
Train on 1919952 samples, validate on 0 samples, 3750 steps per epoch
FM      SIN FUGA  NDCG@10=0.905  MAP@10=0.843  [1500s]
cuda
Train on 1919952 samples, validate on 0 samples, 3750 steps per epoch
cuda
Train on 1919952 samples, validate on 0 samples, 3750 steps per epoch
cuda
Train on 1919952 samples, validate on 0 samples, 3750 steps per epoch
DeepFM  SIN FUGA  NDCG@10=0.909  MAP@10=0.849  [1659s]
cuda
Train on 1919952 samples, validate on 0 samples, 3750 steps per epoch
cuda
Train on 1919952 samples, validate on 0 samples, 3750 steps per epoch
cuda
Train on 1919952 samples, validate on 0 samples, 3750 steps per epoch
DeepNN  SIN FUGA  NDCG@10=0.925  MAP@10=0.873  [1264s]


## 7. Tabla final — con fuga vs sin fuga

In [11]:
# Tabla final — Kozyriev (robustness del protocolo leaky de Cheuque)
rows = []
rows.append(['ALS (full-ranking, 5-fold)', f'{als_ndcg:.4f}', '-', f'{als_map:.4f}'])
for k in ['FM', 'DeepFM', 'DeepNN']:
    rows.append([k + ' (per-user, leaky)', f'{deep_res[k][0]:.4f}', f'{deep_res_noleak[k][0]:.4f}', f'{deep_res[k][1]:.4f}'])
tab = pd.DataFrame(rows, columns=['Modelo', 'NDCG@10 (con fuga)', 'NDCG@10 (sin fuga)', 'MAP@10 (con fuga)'])
print(tab.to_string(index=False))
print('\nUCSD/Cheuque ref (otro dataset): ALS NDCG@10 ~0.33 | deep con fuga ~0.94-0.99 | deep sin fuga ~0.73')
print('Lectura: si el deep leaky vuelve a dispararse (~0.9+) y CAE al quitar la fuga, el ~0.99 es')
print('ARTEFACTO DE PROTOCOLO (label binario + re-ranking por-usuario + fuga de playtime), no del dataset.')
tab

                    Modelo NDCG@10 (con fuga) NDCG@10 (sin fuga) MAP@10 (con fuga)
ALS (full-ranking, 5-fold)             0.1403                  -            0.0624
      FM (per-user, leaky)             0.9985             0.9051            0.9983
  DeepFM (per-user, leaky)             0.9976             0.9090            0.9967
  DeepNN (per-user, leaky)             0.9985             0.9249            0.9983

UCSD/Cheuque ref (otro dataset): ALS NDCG@10 ~0.33 | deep con fuga ~0.94-0.99 | deep sin fuga ~0.73
Lectura: si el deep leaky vuelve a dispararse (~0.9+) y CAE al quitar la fuga, el ~0.99 es
ARTEFACTO DE PROTOCOLO (label binario + re-ranking por-usuario + fuga de playtime), no del dataset.


,Modelo,NDCG@10 (con fuga),NDCG@10 (sin fuga),MAP@10 (con fuga)
0,"ALS (full-ranking, 5-fold)",0.1403,-,0.0624
1,"FM (per-user, leaky)",0.9985,0.9051,0.9983
2,"DeepFM (per-user, leaky)",0.9976,0.9090,0.9967
3,"DeepNN (per-user, leaky)",0.9985,0.9249,0.9983


## 8. Notas de reproducibilidad\n\n- **Robustness check:** si el deep leaky reaparece ~0,9+ y cae sin fuga, el inflado de Cheuque es de PROTOCOLO, no de UCSD.\n- **Filtro:** Kozyriev es review-based (no ownership) → el ≥100/usuario de Cheuque puede colapsar; ver el umbral efectivamente usado en la celda 2 (con fallback declarado).\n- **ALS** factors=500/iters=300/5-fold es lo lento; reducir vía env `CHEUQUE_ALS_FACTORS` / `CHEUQUE_ALS_ITERS` / `CHEUQUE_ALS_FOLDS` para un sanity rápido.